## 0. Kaggle bootstrap

Run this cell **first** on Kaggle. It clones the fork (for `src/`), installs
`pytorch-msssim` / `torchinfo`, links your uploaded `dirty.npy` / `clean.npy`
Dataset as `./data`, and `chdir`s into `notebooks/` so the rest of the
notebook's `../data` and `../results` paths resolve. On a local machine it is a no-op.

**Kaggle setup:** Settings → Accelerator = *GPU*, Internet = *On*; then
*Add Input* → your uploaded data Dataset.


In [ ]:
# >>> KAGGLE BOOTSTRAP >>>  (no-op when not running on Kaggle)
import os, sys, subprocess, glob

if os.path.exists('/kaggle'):
    REPO_URL = 'https://github.com/KrishanYadav333/EXXA.git'
    BRANCH   = 'week-4'
    REPO     = '/kaggle/working/EXXA'
    PKG      = os.path.join(REPO, 'DENOISING_DIFFUSION')  # contains src/, notebooks/

    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO], check=True)

    # deps not guaranteed on the Kaggle image
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim', 'torchinfo'], check=True)

    # expose the uploaded Kaggle Dataset as <repo>/data so '../data/*.npy' resolves
    os.makedirs(os.path.join(PKG, 'data'), exist_ok=True)
    hits = glob.glob('/kaggle/input/**/dirty.npy', recursive=True)
    if hits:
        src_dir = os.path.dirname(hits[0])
        for fn in ('dirty.npy', 'clean.npy'):
            dst = os.path.join(PKG, 'data', fn)
            if not os.path.exists(dst):
                try:
                    os.symlink(os.path.join(src_dir, fn), dst)
                except OSError:
                    import shutil; shutil.copy(os.path.join(src_dir, fn), dst)
    else:
        print('WARNING: dirty.npy not found under /kaggle/input — use *Add Input* to attach your data Dataset.')

    # run from notebooks/ so the existing '..'-relative paths work
    os.chdir(os.path.join(PKG, 'notebooks'))
    if PKG not in sys.path:
        sys.path.insert(0, PKG)

print('cwd :', os.getcwd())
# <<< KAGGLE BOOTSTRAP <<<

# 03 VAE Model — Variational Autoencoder for Astronomical Image Denoising

Continues from `02_autoencoder_model.ipynb`.  
We now train a **Variational Autoencoder (VAE)** on the same 975 protoplanetary disk pairs.

The VAE extends the deterministic autoencoder with a probabilistic latent space:
- **Encoder** outputs μ and log σ² maps instead of a single bottleneck feature map
- **Reparameterization trick** samples z = μ + ε·σ (differentiable)
- **Decoder** reconstructs the clean image from z
- **Loss** = 0.8·MSE + 0.2·(1−SSIM) + 0.001·KL


## 1. Setup

In [1]:
import sys, os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

# Confirm GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Device : cuda
GPU    : NVIDIA GeForce RTX 2050
VRAM   : 4.3 GB


## 2. VAE Architecture

### ConvBlock
Two 3×3 conv layers with BatchNorm + ReLU. Same building block as the autoencoder.

### Encoder
```
Input  (B, 1, 64, 64)
enc1   (B, 32, 64, 64)   ConvBlock(1, 32)
pool   (B, 32, 32, 32)   MaxPool2d
enc2   (B, 64, 32, 32)   ConvBlock(32, 64)
pool   (B, 64, 16, 16)
enc3   (B,128, 16, 16)   ConvBlock(64, 128)
pool   (B,128,  8,  8)
pre    (B,256,  8,  8)   ConvBlock(128, 256)  ← shared trunk
mu     (B,128,  8,  8)   Conv2d(256, 128, 1)  ← distribution mean
log_var(B,128,  8,  8)   Conv2d(256, 128, 1)  ← distribution log-variance
```

### Reparameterization trick
```
z = mu + eps * exp(0.5 * log_var),   eps ~ N(0, I)
```
Allows gradients to flow back through mu and log_var even though z is sampled.

### Decoder
```
z    (B,128,  8,  8)
up3  (B,128, 16, 16)   ConvTranspose2d
dec3 (B,128, 16, 16)
up2  (B, 64, 32, 32)
dec2 (B, 64, 32, 32)
up1  (B, 32, 64, 64)
dec1 (B, 32, 64, 64)
out  (B,  1, 64, 64)   Conv2d + Sigmoid → [0, 1]
```


In [2]:
from src.models.vae import DenoisingVAE

model = DenoisingVAE(latent_dim=128).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'DenoisingVAE parameters : {total_params:,}')

# Verify forward pass shapes
x_test = torch.randn(2, 1, 64, 64).to(device)
with torch.no_grad():
    out_test, mu_test, lv_test = model(x_test)

print(f'Input   : {x_test.shape}')
print(f'Output  : {out_test.shape}')
print(f'mu      : {mu_test.shape}')
print(f'log_var : {lv_test.shape}')
print(f'Output range : [{out_test.min():.3f}, {out_test.max():.3f}]  (expected 0-1)')


DenoisingVAE parameters : 1,734,561


Input   : torch.Size([2, 1, 64, 64])
Output  : torch.Size([2, 1, 64, 64])
mu      : torch.Size([2, 128, 8, 8])
log_var : torch.Size([2, 128, 8, 8])
Output range : [0.096, 0.720]  (expected 0-1)


## 3. VAE Loss Function — MSE + SSIM + KL

```
total = 0.8 × MSE + 0.2 × (1 − SSIM) + 0.001 × KL
```

| Term | Role | Weight |
|------|------|--------|
| MSE  | Pixel-level accuracy | 0.8 |
| 1−SSIM | Structural/perceptual quality | 0.2 |
| KL   | Latent space regularisation → forces z near N(0,I) | 0.001 |

**KL divergence** for a Gaussian posterior:
```
KL = −0.5 × mean(1 + log_var − mu² − exp(log_var))
```
The small gamma=0.001 lets reconstruction dominate early training  
while the KL prevents posterior collapse.


In [3]:
from src.utils.losses import VAELoss

criterion = VAELoss(alpha=0.8, beta=0.2, gamma=0.001)
print(f'alpha={criterion.alpha}  beta={criterion.beta}  gamma={criterion.gamma}')

# Sanity check with random tensors
torch.manual_seed(42)
p  = torch.rand(2, 1, 64, 64)
gt = torch.rand(2, 1, 64, 64)
mu_s  = torch.randn(2, 128, 8, 8)
lv_s  = torch.randn(2, 128, 8, 8)

total, mse, ssim_l, kl = criterion(p, gt, mu_s, lv_s)
print(f'Total : {total.item():.6f}')
print(f'MSE   : {mse.item():.6f}')
print(f'SSIM  : {ssim_l.item():.6f}  (1 - SSIM score)')
print(f'KL    : {kl.item():.6f}')


alpha=0.8  beta=0.2  gamma=0.001
Total : 0.333165
MSE   : 0.166608
SSIM  : 0.995205  (1 - SSIM score)
KL    : 0.837788


## 4. Data Pipeline

In [4]:
SEED       = 42
PATCH_SIZE = 64
BATCH_SIZE = 16
GRAD_ACCUM = 4        # effective batch = 16 * 4 = 64
LR         = 1e-3
EPOCHS     = 30

np.random.seed(SEED)
torch.manual_seed(SEED)

dirty_all = np.load('../data/dirty.npy').astype(np.float32)
clean_all = np.load('../data/clean.npy').astype(np.float32)
print(f'dirty : {dirty_all.shape}  clean : {clean_all.shape}')

indices = np.arange(len(dirty_all))
train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=SEED)
print(f'Train : {len(train_idx)}  Val : {len(val_idx)}')


class PatchDataset(Dataset):
    def __init__(self, dirty, clean, idx, ps=64):
        self.dirty, self.clean, self.idx, self.ps = dirty, clean, idx, ps
        self._h, self._w = dirty.shape[1], dirty.shape[2]

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        idx = self.idx[i]
        r = np.random.randint(0, self._h - self.ps + 1)
        c = np.random.randint(0, self._w - self.ps + 1)
        dp = self.dirty[idx, r:r+self.ps, c:c+self.ps]
        cp = self.clean[idx, r:r+self.ps, c:c+self.ps]
        lo, hi = dp.min(), dp.max()
        if hi > lo:
            dp = (dp - lo) / (hi - lo)
            cp = np.clip((cp - lo) / (hi - lo), 0.0, 1.0)
        return torch.from_numpy(dp[np.newaxis]), torch.from_numpy(cp[np.newaxis])


train_loader = DataLoader(PatchDataset(dirty_all, clean_all, train_idx),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(PatchDataset(dirty_all, clean_all, val_idx),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
print(f'Train batches : {len(train_loader)}  Val batches : {len(val_loader)}')
print(f'Effective batch size (grad accum x{GRAD_ACCUM}) : {BATCH_SIZE * GRAD_ACCUM}')


dirty : (975, 600, 600)  clean : (975, 600, 600)
Train : 780  Val : 195
Train batches : 49  Val batches : 13
Effective batch size (grad accum x4) : 64


## 5. Training Loop — 30 Epochs with Gradient Accumulation

**Gradient accumulation** (steps=4) simulates a larger effective batch (64 samples)  
without increasing VRAM usage. Gradients are accumulated across 4 mini-batches  
before each `optimizer.step()`.

Columns printed each epoch:
- `ep` : epoch number
- `total` : full VAE loss (train / val)
- `mse` : MSE sub-component (train)
- `ssim` : 1−SSIM sub-component (train)
- `kl` : KL divergence sub-component (train)


In [5]:
# Fresh model + optimizer
model_vae = DenoisingVAE(latent_dim=128).to(device)
criterion = VAELoss(alpha=0.8, beta=0.2, gamma=0.001)
optimizer = torch.optim.Adam(model_vae.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

best_val_loss = float('inf')
best_epoch    = -1
train_history = []
val_history   = []

hdr = f"{'ep':>3}  {'tr_total':>10}  {'tr_mse':>10}  {'tr_ssim':>10}  {'tr_kl':>10}  {'val_total':>10}  {'lr':>8}"
print(hdr)
print('-' * len(hdr))

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model_vae.train()
    tr_total = tr_mse = tr_ssim = tr_kl = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (dirty, clean) in enumerate(train_loader, 1):
        dirty = dirty.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)

        recon, mu, log_var = model_vae(dirty)
        total, mse, ssim_l, kl = criterion(recon, clean, mu, log_var)

        # Scale loss by accumulation steps so gradients are averaged
        (total / GRAD_ACCUM).backward()

        n = dirty.size(0)
        tr_total += total.item() * n
        tr_mse   += mse.item()   * n
        tr_ssim  += ssim_l.item()* n
        tr_kl    += kl.item()    * n

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

    N = len(train_loader.dataset)
    tr_total /= N; tr_mse /= N; tr_ssim /= N; tr_kl /= N

    # Validation
    model_vae.eval()
    vl = 0.0
    with torch.no_grad():
        for dirty, clean in val_loader:
            dirty = dirty.to(device, non_blocking=True)
            clean = clean.to(device, non_blocking=True)
            recon, mu, log_var = model_vae(dirty)
            total, _, _, _ = criterion(recon, clean, mu, log_var)
            vl += total.item() * dirty.size(0)
    vl /= len(val_loader.dataset)

    scheduler.step(vl)
    current_lr = optimizer.param_groups[0]['lr']

    train_history.append(tr_total)
    val_history.append(vl)

    improved = ' <<' if vl < best_val_loss else ''
    if vl < best_val_loss:
        best_val_loss = vl
        best_epoch    = epoch
        best_state    = {k: v.clone() for k, v in model_vae.state_dict().items()}

    elapsed = time.time() - t0
    print(f'{epoch:>3}  {tr_total:>10.6f}  {tr_mse:>10.6f}  {tr_ssim:>10.6f}  '
          f'{tr_kl:>10.6f}  {vl:>10.6f}  {current_lr:>8.2e}  {elapsed:.1f}s{improved}')

print(f'\nBest val loss : {best_val_loss:.6f} at epoch {best_epoch}')


 ep    tr_total      tr_mse     tr_ssim       tr_kl   val_total        lr
-------------------------------------------------------------------------


  1    0.254554    0.118372    0.797174    0.421357    0.269406  1.00e-03  2.5s <<


  2    0.182481    0.060887    0.666748    0.422283    0.197680  1.00e-03  1.7s <<


  3    0.155437    0.038396    0.621257    0.468705    0.167064  1.00e-03  1.7s <<


  4    0.141872    0.027794    0.595585    0.519978    0.148588  1.00e-03  1.7s <<


...

 26    0.049485    0.011835    0.197359    0.545438    0.117249  1.00e-03  1.9s


 27    0.052020    0.011568    0.210865    0.592508    0.039117  1.00e-03  1.9s <<


 28    0.049377    0.011268    0.198966    0.568898    0.054731  1.00e-03  1.9s


 29    0.049150    0.011637    0.196585    0.523252    0.050405  1.00e-03  2.0s


 30    0.049044    0.012912    0.190957    0.522562    0.046897  1.00e-03  1.9s

Best val loss : 0.039117 at epoch 27


## 6. Save Best Checkpoint

In [6]:
os.makedirs('../results/checkpoints', exist_ok=True)
ckpt_path = '../results/checkpoints/vae_best.pth'

# Reload best weights into model_vae
model_vae.load_state_dict(best_state)
model_vae.eval()

torch.save({
    'epoch'               : best_epoch,
    'model_state_dict'    : model_vae.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'val_loss'            : best_val_loss,
    'alpha'               : 0.8,
    'beta'                : 0.2,
    'gamma'               : 0.001,
    'latent_dim'          : 128,
    'arch'                : 'DenoisingVAE',
}, ckpt_path)

print(f'Checkpoint saved -> {ckpt_path}')
print(f'Best epoch : {best_epoch}  |  Best val loss : {best_val_loss:.6f}')


Checkpoint saved -> ../results/checkpoints/vae_best.pth
Best epoch : 27  |  Best val loss : 0.039117


### Loss Curves

In [7]:
fig, ax = plt.subplots(figsize=(9, 5))
eps = range(1, EPOCHS + 1)
ax.plot(eps, train_history, label='Train VAE Loss', linewidth=2, color='#4C9EEB')
ax.plot(eps, val_history,   label='Val VAE Loss',   linewidth=2, color='#E8715A', linestyle='--')
ax.axvline(best_epoch, color='gray', linestyle=':', linewidth=1.2,
           label=f'Best epoch ({best_epoch})')
ax.scatter([best_epoch], [best_val_loss], color='#E8715A', zorder=5, s=80)
ax.annotate(f'Best: {best_val_loss:.5f}', xy=(best_epoch, best_val_loss),
            xytext=(best_epoch + 0.8, best_val_loss + 2e-4), fontsize=9, color='#E8715A')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('VAE Loss (0.8×MSE + 0.2×SSIM + 0.001×KL)', fontsize=10)
ax.set_title('DenoisingVAE -- Training Curves', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout()

os.makedirs('../results', exist_ok=True)
plt.savefig('../results/vae_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> ../results/vae_loss.png')


Saved -> ../results/vae_loss.png


C:\Users\kk456\AppData\Local\Temp\ipykernel_8000\3883902362.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Visual Comparison — VAE vs Hybrid AE vs Clean GT vs Noisy Input

4 columns:
1. **Dirty** — noisy interferometric observation
2. **Clean GT** — ground truth from simulation
3. **Hybrid AE** (30 ep, MSE-only best checkpoint from notebook 02)
4. **VAE** (30 ep, best checkpoint from this notebook)


In [8]:
from src.models.autoencoder import DenoisingAutoencoder

# Load hybrid autoencoder checkpoint
model_ae = DenoisingAutoencoder().to(device)
ckpt_ae  = torch.load('../results/checkpoints/autoencoder_hybrid_best.pth', map_location=device)
model_ae.load_state_dict(ckpt_ae['model_state_dict'])
model_ae.eval()
print('Loaded hybrid AE checkpoint  (epoch', ckpt_ae.get('epoch', '?'), ')')

# model_vae already holds best weights from training above
model_vae.eval()
ckpt_vae = torch.load(ckpt_path, map_location=device)
print('Loaded VAE checkpoint         (epoch', ckpt_vae['epoch'], ')')


Loaded hybrid AE checkpoint  (epoch 27 )


Loaded VAE checkpoint         (epoch 27 )


In [9]:
rng = np.random.default_rng(99)
ids = rng.integers(0, len(dirty_all), size=3)
r, c = 268, 268   # center 64x64 crop

cols  = ['Dirty', 'Clean GT', 'Hybrid AE (30 ep)', 'VAE (30 ep)']
fig, axes = plt.subplots(3, 4, figsize=(14, 10))

for col_idx, title in enumerate(cols):
    axes[0, col_idx].set_title(title, fontsize=11, fontweight='bold')

for row, idx in enumerate(ids):
    dp = dirty_all[idx, r:r+64, c:c+64]
    cp = clean_all[idx, r:r+64, c:c+64]
    lo, hi = dp.min(), dp.max()
    dp_n = (dp - lo) / (hi - lo) if hi > lo else dp
    cp_n = np.clip((cp - lo) / (hi - lo), 0, 1) if hi > lo else cp

    inp = torch.from_numpy(dp_n[np.newaxis, np.newaxis]).to(device)
    with torch.no_grad():
        pred_ae  = model_ae(inp).squeeze().cpu().numpy()
        pred_vae, _, _ = model_vae(inp)
        pred_vae = pred_vae.squeeze().cpu().numpy()

    for col_idx, img in enumerate([dp_n, cp_n, pred_ae, pred_vae]):
        axes[row, col_idx].imshow(img, cmap='inferno', vmin=0, vmax=1)
        axes[row, col_idx].axis('off')
    axes[row, 0].set_ylabel(f'#{idx}', fontsize=10)

plt.suptitle(
    'VAE vs Hybrid AE vs Ground Truth — center 64x64 crop (inferno colormap)',
    fontweight='bold', fontsize=12
)
plt.tight_layout()
os.makedirs('../experiments', exist_ok=True)
plt.savefig('../experiments/vae_vs_ae_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> ../experiments/vae_vs_ae_comparison.png')


Saved -> ../experiments/vae_vs_ae_comparison.png


C:\Users\kk456\AppData\Local\Temp\ipykernel_8000\960996901.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Summary

| Model | Val Loss (type) | Notes |
|-------|----------------|-------|
| Noisy input | MSE = 0.009048 | baseline |
| Gaussian sigma=2 | MSE = 0.006806 | best classical |
| Autoencoder MSE-only (30 ep) | MSE = 0.007239 | deterministic |
| Autoencoder HybridLoss (30 ep) | MSE = 0.007181 | MSE + SSIM |
| **VAE HybridLoss+KL (30 ep)** | *see above* | **MSE + SSIM + KL** |

### What the VAE adds over the deterministic autoencoder
- **Probabilistic latent space** — each input maps to a distribution, not a point
- **KL regularisation** — prevents the encoder from ignoring the prior, improves generalisation
- **Structured latent space** — enables interpolation and sampling (useful for future DDPM conditioning)

### Next steps
- U-Net with skip connections (better structural preservation)
- DDPM denoising diffusion model


## 9. Final Unified Evaluation — All Methods on 100 Val Samples

Evaluates every method on the **same 100 validation images** using
**consistent per-patch normalisation** so neural and classical metrics
are directly comparable.

Evaluation protocol
- Images are evaluated as full 600×600 arrays
- Classical filters run on raw full images
- Neural models use a sliding-window (patch=64, stride=32, 50 % overlap)
  with per-patch normalisation identical to training
- Metrics: PSNR (dB), SSIM, MSE — averaged over 100 samples
- Results saved to `results/metrics_final.csv`

> **Key metric for this task:** SSIM — higher means better disk-structure
> preservation. PSNR/MSE slightly favour classical filters due to their
> blurring behaviour, but at the cost of smearing disk gaps.


In [10]:
import os, time
import numpy as np
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.ndimage  import gaussian_filter, median_filter
from scipy.signal   import wiener as wiener_filter
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity   as ssim_fn
from src.models.autoencoder import DenoisingAutoencoder
from src.models.vae         import DenoisingVAE

# ── Config ────────────────────────────────────────────────────────────────────
SEED        = 42
N_SAMPLES   = 100
PATCH_SIZE  = 64
STRIDE      = 32
BATCH_INFER = 32

np.random.seed(SEED); torch.manual_seed(SEED)

# ── Load data ─────────────────────────────────────────────────────────────────
dirty_all = np.load('../data/dirty.npy').astype(np.float32)
clean_all = np.load('../data/clean.npy').astype(np.float32)

_, val_idx = train_test_split(np.arange(len(dirty_all)), test_size=0.20, random_state=SEED)
rng     = np.random.default_rng(SEED)
chosen  = np.sort(rng.choice(val_idx, size=min(N_SAMPLES, len(val_idx)), replace=False))
print(f"Evaluating {len(chosen)} val samples  (idx {chosen[0]}..{chosen[-1]})")

# ── Load neural checkpoints ───────────────────────────────────────────────────
def load_ae(name):
    m = DenoisingAutoencoder().to(device)
    ck = torch.load(f'../results/checkpoints/{name}', map_location=device)
    m.load_state_dict(ck['model_state_dict']); m.eval(); return m

def load_vae_ck(name):
    ck = torch.load(f'../results/checkpoints/{name}', map_location=device)
    m  = DenoisingVAE(latent_dim=ck.get('latent_dim', 128)).to(device)
    m.load_state_dict(ck['model_state_dict']); m.eval(); return m

ae_mse    = load_ae('autoencoder_best.pth')
ae_hybrid = load_ae('autoencoder_hybrid_best.pth')
vae_m     = load_vae_ck('vae_best.pth')
print("Checkpoints loaded.")

# ── Sliding-window inference ──────────────────────────────────────────────────
def nn_denoise(model, dirty_img, is_vae=False):
    H, W = dirty_img.shape
    out_sum = np.zeros((H, W), np.float64)
    out_cnt = np.zeros((H, W), np.float64)

    rows = list(range(0, H - PATCH_SIZE + 1, STRIDE))
    cols = list(range(0, W - PATCH_SIZE + 1, STRIDE))
    if rows[-1] + PATCH_SIZE < H: rows.append(H - PATCH_SIZE)
    if cols[-1] + PATCH_SIZE < W: cols.append(W - PATCH_SIZE)

    patches, positions = [], []
    for r in rows:
        for c in cols:
            p = dirty_img[r:r+PATCH_SIZE, c:c+PATCH_SIZE].copy()
            lo, hi = p.min(), p.max()
            p = (p - lo) / (hi - lo) if hi > lo else p
            patches.append(p); positions.append((r, c))

    model.eval()
    preds = []
    with torch.no_grad():
        for s in range(0, len(patches), BATCH_INFER):
            b = torch.from_numpy(np.stack(patches[s:s+BATCH_INFER])[:, np.newaxis]).to(device)
            out = model(b)
            if is_vae: out = out[0]
            preds.extend(out.squeeze(1).cpu().numpy())

    for pred, (r, c) in zip(preds, positions):
        out_sum[r:r+PATCH_SIZE, c:c+PATCH_SIZE] += pred
        out_cnt[r:r+PATCH_SIZE, c:c+PATCH_SIZE] += 1.0

    return np.clip(out_sum / np.maximum(out_cnt, 1e-8), 0, 1).astype(np.float32)

# ── Metric helper ─────────────────────────────────────────────────────────────
def calc(clean, denoised):
    d = np.clip(denoised.astype(np.float32), 0, 1)
    return {
        'PSNR': psnr_fn(clean, d, data_range=1.0),
        'SSIM': ssim_fn(clean, d, data_range=1.0),
        'MSE' : float(np.mean((clean - d)**2)),
    }

# ── Accumulate ────────────────────────────────────────────────────────────────
methods = ['Noisy input', 'Gaussian s=2', 'Median 3x3', 'Wiener',
           'AE MSE-only', 'AE HybridLoss', 'VAE (MSE+SSIM+KL)']
acc = {m: {'PSNR': [], 'SSIM': [], 'MSE': []} for m in methods}

t0 = time.time()
for i, idx in enumerate(chosen):
    c = clean_all[idx]; d = dirty_all[idx]

    def push(name, arr):
        r = calc(c, arr)
        for k in ('PSNR','SSIM','MSE'): acc[name][k].append(r[k])

    push('Noisy input',        d)
    push('Gaussian s=2',       gaussian_filter(d, sigma=2.0))
    push('Median 3x3',         median_filter(d, size=3))
    push('Wiener',             wiener_filter(d).astype(np.float32))
    push('AE MSE-only',        nn_denoise(ae_mse,    d))
    push('AE HybridLoss',      nn_denoise(ae_hybrid, d))
    push('VAE (MSE+SSIM+KL)',  nn_denoise(vae_m,     d, is_vae=True))

    if (i+1) % 20 == 0:
        print(f"  [{i+1:>3}/100]  {time.time()-t0:.0f}s")

print(f"Done in {time.time()-t0:.1f}s")

# ── Build DataFrame ───────────────────────────────────────────────────────────
rows = []
for m in methods:
    rows.append({
        'Method': m,
        'PSNR':   round(np.mean(acc[m]['PSNR']), 4),
        'SSIM':   round(np.mean(acc[m]['SSIM']), 4),
        'MSE':    round(np.mean(acc[m]['MSE']),  6),
    })

df = pd.DataFrame(rows).sort_values('SSIM', ascending=False).reset_index(drop=True)
df.index += 1   # 1-based rank by SSIM

# Save CSV
os.makedirs('../results', exist_ok=True)
csv_path = '../results/metrics_final.csv'
df.to_csv(csv_path, index_label='Rank')
print(f"Saved -> {csv_path}")

# ── Pretty-print table ────────────────────────────────────────────────────────
print(f"\n{'='*58}")
print(f"  Unified Comparison — {len(chosen)} Val Samples  (ranked by SSIM)")
print(f"{'='*58}")
print(f"  {'Method':<24} {'PSNR':>8}  {'SSIM':>6}  {'MSE':>10}")
print(f"  {'-'*54}")
best_psnr = df['PSNR'].max(); best_ssim = df['SSIM'].max(); best_mse = df['MSE'].min()
for _, row in df.iterrows():
    p_s = '  *' if row['PSNR']==best_psnr else '   '
    s_s = '  *' if row['SSIM']==best_ssim else '   '
    m_s = '  *' if row['MSE'] ==best_mse  else '   '
    print(f"  {row['Method']:<24} {row['PSNR']:>8.4f}{p_s} {row['SSIM']:>6.4f}{s_s} {row['MSE']:>10.6f}{m_s}")
print(f"{'='*58}")
print("  * = best in column")
print(f"\nNote: SSIM is the primary quality metric for this task.")
print(f"      PSNR/MSE favour blurring; SSIM measures structure fidelity.")


Evaluating 100 val samples  (idx 29..973)


Checkpoints loaded.


  [ 20/100]  21s


  [ 40/100]  38s


  [ 60/100]  56s


  [ 80/100]  79s


  [100/100]  102s
Done in 101.8s
Saved -> ../results/metrics_final.csv

  Unified Comparison — 100 Val Samples  (ranked by SSIM)
  Method                       PSNR    SSIM         MSE
  ------------------------------------------------------
  AE HybridLoss             19.9152    0.7609  *   0.013920   
  VAE (MSE+SSIM+KL)         19.9951    0.7059      0.013336   
  AE MSE-only               20.2513    0.6158      0.012804   
  Gaussian s=2              22.7803    0.4230      0.006380   
  Median 3x3                22.8835  * 0.3591      0.006317  *
  Wiener                    22.5687    0.3398      0.006702   
  Noisy input               21.5703    0.1924      0.008391   
  * = best in column

Note: SSIM is the primary quality metric for this task.
      PSNR/MSE favour blurring; SSIM measures structure fidelity.
